In [1]:
import os
import yaml
import time
from statistics import mean
import matplotlib.pyplot as plt
import scipy.io

from rnn_models import *
from transformer_model import *
from outlier_detection import detect_outlier
from transformations import *
from task_funcs import *

# %matplotlib notebook

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Cuda available: ", torch.cuda.is_available())
# program seed and model seed
seed = 321
random.seed(seed)

# Model Settings
BATCH_SIZE = 64
MAX_EPOCHS = 5000
ENABLE_ROTATION_AUG = True
ENABLE_SCALING_AUG = False
PRINT_INTERVAL = 10
MODEL_DIR_PATH = "./models_4tasks_fixed_compact/"
GRAPH_DIR_PATH = "./graphs/"
LOG_DIR = './logs'  # saved trajectory directory
DATA_DIR = '../Process_data/postprocessed'
SAVED_TRAJ_VER_SECONDARY = "saved_traj_9-10-23" # saved trajectory subdirectory (non-transformer models)

MODEL_COPIES = 3 # model copies for training different model on different unique demos
TOTAL_DEMOS_PER_TASK = 30
TRAJ_SAVE_DIR = "extended_task_trajs"
# Task Information
# Model copy number, also determine which subset of demos to use
kth, n_train, only_ith_task = 0, 30, -1
task_files = ["2022-10-06", "2022-10-27", "2022-12-01", "2023-03-17"]
task_names = ["Pick&Place", "Pouring", "Shooting", "Sweeping"]
task_old_names = ["extrapolation", "pouring", "shooting", "sweeping"]
task_dims = ['x', 'y', 'z', 'qx', 'qy', 'qz', 'qw']
start_obj_index, end_obj_index = [0,0,1,0], [1,1,0,1]
n_dims = len(task_dims)
n_demos_list = [5, 10, 20, 30]
max_len = 128

# Available models for analysis
main_models, secondary_models = ["TP-Tf", "LSTM_MSE"], ['tp-gmm', 'tp-pmp', 'kmp']
running_model = "TP-Tf"

# Model and trajectory locations
DIR_SUFFIX = f"rot{1 if ENABLE_ROTATION_AUG else 0}_scale{1 if ENABLE_SCALING_AUG else 0}"
if only_ith_task >= 0:
    DIR_SUFFIX = f"{DIR_SUFFIX}_{task_files[only_ith_task]}"
MODEL_SUB_DIR = f"{running_model}nodq_{DIR_SUFFIX}"
# Find all objects first
all_objs = []
selected_tasks = [task for task in task_files if only_ith_task==-1 or task==task_files[only_ith_task]]
n_tasks = len(selected_tasks)
for file in selected_tasks:
    task_config_dir = os.path.join(DATA_DIR, file)
    with open(os.path.join(task_config_dir, 'task_config.yaml')) as file:
        config = yaml.load(file, Loader=yaml.FullLoader)
    objs = config["individuals"]
    all_objs = all_objs + objs
# Instaniate all object type and enable/disable via-points
unique_objs = ['trajectory'] + sorted(list(set(all_objs)))
n_objs = len(unique_objs)
obj_tags = create_tags(unique_objs) # tags created based on unique objects and added later on to object sequences
task_tags = create_tags(selected_tasks)

train_objs_pos, train_traj_pos = [], []
valid_objs_pos, valid_traj_pos = [], []
test_objs_pos, test_traj_pos = [], []

all_test_demos = []
test_splits = []
# Load all task data and create data test/training splits
for task_file in selected_tasks:
    task_config_dir = os.path.join(DATA_DIR, task_file)
    with open(os.path.join(task_config_dir, 'task_config.yaml')) as file:
        config = yaml.load(file, Loader=yaml.FullLoader)
    project_dir = config["project_path"] # Modify this to your need
    base_dir = os.path.join(project_dir, config["postprocessed_dir"])
    triangulation = 'dlc3d'
    template_dir = os.path.join(project_dir, config["postprocessed_dir"],f'transformations/{triangulation}')
    individuals = config["individuals"] # The objects that we will place a reference frame on
    with open(os.path.join(base_dir, 'processed', triangulation, 'gripper_trajs_in_obj_aligned_filtered.pickle',), 'rb') as f1:
        gripper_trajs_in_obj_all_actions = pickle.load(f1)
    with open(os.path.join(base_dir, 'processed', triangulation, 'HTs_obj_in_ndi.pickle',), 'rb') as f2:
        HTs_obj_in_ndi_all_actions = pickle.load(f2)
    with open(os.path.join(base_dir, 'processed', triangulation, 'gripper_traj_in_grouped_objs_aligned_filtered.pickle',), 'rb') as f3:
        gripper_trajs_in_grouped_objs_all_actions = pickle.load(f3)
    with open(os.path.join(base_dir, 'processed', triangulation, 'HTs_grouped_objs_in_ndi.pickle',), 'rb') as f4:
        HTs_grouped_objs_in_ndi_all_actions = pickle.load(f4)

    ind = 0  # index of action to be tested
    # gripper_trajs_in_ndi = gripper_trajs_truncated[ind]
    gripper_traj_in_obj = gripper_trajs_in_obj_all_actions[ind]
    gripper_traj_in_grouped_obj = gripper_trajs_in_grouped_objs_all_actions[ind]
    gripper_traj_in_generalized_obj = gripper_traj_in_obj | gripper_traj_in_grouped_obj

    HTs_obj_in_ndi = HTs_obj_in_ndi_all_actions[ind]
    HTs_grouped_obj_in_ndi = HTs_grouped_objs_in_ndi_all_actions[ind]
    if not 'global' in HTs_obj_in_ndi.keys():
        HTs_obj_in_ndi = swap_dict_level(HTs_obj_in_ndi)
        HTs_grouped_obj_in_ndi = swap_dict_level(HTs_grouped_obj_in_ndi)
    HTs_generalized_obj_in_ndi = HTs_obj_in_ndi | HTs_grouped_obj_in_ndi

    # Remove outliers
    outliers = []
    std_thres = 3
    for individual in individuals:
        n_std = std_thres
        outlier_individual = detect_outlier(gripper_traj_in_generalized_obj[individual], n=n_std)
        print(f'The outliers for individual {individual} are {outlier_individual}')
        outliers += outlier_individual
    outliers = list(set(outliers))
    bad_demos = outliers
    demos = sorted(list(HTs_generalized_obj_in_ndi['global'].keys()))
    demos = [demo for demo in demos if demo not in bad_demos]

    saved_demo_file = os.path.join(LOG_DIR, "saved_demos_30_4tasks.pkl")
    with open(saved_demo_file, 'rb') as fout:
        saved_demos = pickle.load(fout)
    train_demos_pool = saved_demos[task_file]["train"]
#     train_demos_pool = [demo for demo in random.sample(demos, 30)]
#     random.shuffle(train_demos_pool)

    test_valid_demos_pool_updated = [demo for demo in demos if demo not in train_demos_pool]
    # Select unique training demo indices based on model copy number
    selected_indices = create_chunks_of_indices(n_train, TOTAL_DEMOS_PER_TASK, MODEL_COPIES)[kth]
    train_demos = [val for i, val in enumerate(train_demos_pool) if i in selected_indices]

    # validation and test demo split
    # split_size = int(len(test_valid_demos_pool_updated)/2)
    # valid_demos = test_valid_demos_pool_updated[:split_size]
    # test_demos = test_valid_demos_pool_updated[split_size:]
    valid_demos = saved_demos[task_file]["valid"]
    test_demos = saved_demos[task_file]["test"]
    all_test_demos.extend(test_demos)
    print(f"{task_file}-{individuals}\nTraining Indices: {selected_indices}\nTraining Demos: {train_demos}\nTest Demos: {test_demos}")
    print(f'The number of training pool for task {task_file} is: {len(train_demos_pool)}')
    print(f'The number of outliers is: {len(outliers)} with remaining {len(demos)}')
    print(f'Training/Test Size: {len(train_demos)}/{len(test_demos)}')
    for demo in train_demos:
        traj = gripper_traj_in_obj['global'][demo][task_dims].to_numpy()
        traj_len = traj.shape[0]
        obj_buffer = obj_tags['trajectory'].repeat([traj_len, 1])
        task_buffer = task_tags[task_file].repeat([traj_len, 1])

        new_traj_data = np.concatenate([traj, obj_buffer, task_buffer], axis=1)
        obj_pos_all = []
        for obj_ind in individuals:
            mat = HTs_generalized_obj_in_ndi[obj_ind][demo]
            if n_dims==3:
                obj_pos = np.concatenate([mat[:3,3], obj_tags[obj_ind], task_tags[task_file]])
            else:
                obj_pos = np.concatenate([mat[:3,3], R.from_matrix(mat[:3,:3]).as_quat(), obj_tags[obj_ind], task_tags[task_file]])
            obj_pos_all.append(obj_pos)
        train_objs_pos.append(np.stack(obj_pos_all))
        train_traj_pos.append(new_traj_data)

    for demo in valid_demos:
        traj = gripper_traj_in_obj['global'][demo][task_dims].to_numpy()
        traj_len = traj.shape[0]
        obj_buffer = obj_tags['trajectory'].repeat([traj_len, 1])
        task_buffer = task_tags[task_file].repeat([traj_len, 1])
        new_traj_data = np.concatenate([traj, obj_buffer, task_buffer], axis=1)
        obj_pos_all = []
        for obj_ind in individuals:
            mat = HTs_generalized_obj_in_ndi[obj_ind][demo]
            if n_dims==3:
                obj_pos = np.concatenate([mat[:3,3], obj_tags[obj_ind], task_tags[task_file]])
            else:
                obj_pos = np.concatenate([mat[:3,3], R.from_matrix(mat[:3,:3]).as_quat(), obj_tags[obj_ind], task_tags[task_file]])
            obj_pos_all.append(obj_pos)

        valid_objs_pos.append(np.stack(obj_pos_all))
        valid_traj_pos.append(new_traj_data)

    test_splits.append(len(test_traj_pos))
    for demo in test_demos:
        traj = gripper_traj_in_obj['global'][demo][task_dims].to_numpy()
        traj_len = traj.shape[0]
        obj_buffer = obj_tags['trajectory'].repeat([traj_len, 1])
        task_buffer = task_tags[task_file].repeat([traj_len, 1])
        new_traj_data = np.concatenate([traj, obj_buffer, task_buffer], axis=1)
        obj_pos_all = []
        for ind, obj_ind in enumerate(individuals):
            mat = HTs_generalized_obj_in_ndi[obj_ind][demo]
            if n_dims==3:
                obj_pos = np.concatenate([mat[:3,3], obj_tags[obj_ind], task_tags[task_file]])
            else:
                obj_pos = np.concatenate([mat[:3,3], R.from_matrix(mat[:3,:3]).as_quat(), obj_tags[obj_ind], task_tags[task_file]])
            obj_pos_all.append(obj_pos)

        test_objs_pos.append(np.stack(obj_pos_all))
        test_traj_pos.append(new_traj_data)

if only_ith_task >= 0:
    max_len = test_traj_pos[0].shape[0]
    print("Targeted Task Length:", max_len)

Cuda available:  True


ModuleNotFoundError: No module named 'pandas.core.indexes.numeric'

In [ ]:
# Define the augmentations and transformations
def obj_traj_min_dist(traj, obj_pos):
    n_objs = obj_pos.shape[0]
    dist_to_objs = []
    for i in range(n_objs):
        w = np.linalg.norm(traj[:,:3]-obj_pos[i,:3], axis=1)
        dist_to_objs.append(w)
    stacked_dist = np.stack(dist_to_objs)
    min_dist = np.min(stacked_dist, axis=0)
    return np.max(min_dist)/min_dist

class TrajectoryDataset(Dataset):
    def __init__(self, obj_data, traj_data, transform_dims, transform=[],
                 max_seq_len=128):
        self.traj_data = traj_data
        self.obj_data = obj_data
        self.transform = transform
        self.dims = transform_dims
        self.max_seq_len = max_seq_len
        # Calculate weight
        data_len = len(traj_data)
        self.weights = []
        for i in range(data_len):
            weight = obj_traj_min_dist(traj_data[i], obj_data[i])
            self.weights.append(weight)

    def __getitem__(self, idx):
        traj_data = self.traj_data[idx].copy()
        obj_data = self.obj_data[idx].copy()
        weight_data = self.weights[idx].copy()
        # Transformation process
        for tf_func in self.transform:
            obj_data, traj_data = tf_func(obj_data, traj_data)

        # Pad trajectory sequence
        traj_data = torch.tensor(traj_data)
        obj_data = torch.tensor(obj_data)
        weight_data = torch.tensor(weight_data)

        if traj_data.shape[0] < self.max_seq_len:
            diff = self.max_seq_len - traj_data.shape[0]
            pad = torch.zeros([diff, traj_data.shape[1]])
            traj_data = torch.cat([traj_data, pad])
            weight_pad = torch.zeros(diff)
            weight_data = torch.cat([weight_data, weight_pad])
        traj_hidden = traj_data.clone()
        traj_hidden[:,:self.dims] = 0
        return obj_data, traj_data, traj_hidden, weight_data, idx

    def __len__(self):
        return len(self.obj_data)


# Find training mean and std
contiguous_traj = np.concatenate(train_traj_pos)
train_mean = np.mean(contiguous_traj[:,:3], axis=0)
train_std = np.std(contiguous_traj[:,:3])/3
# Normalization
norm_func = normalize_wrapper(train_mean, train_std)
train_objs_pos = list(map(norm_func, train_objs_pos))
train_traj_pos = list(map(norm_func, train_traj_pos))
valid_objs_pos = list(map(norm_func, valid_objs_pos))
valid_traj_pos = list(map(norm_func, valid_traj_pos))
test_objs_pos = list(map(norm_func, test_objs_pos))
test_traj_pos = list(map(norm_func, test_traj_pos))

# extra transformations for fewer unique demos
size_modifier = int(60/n_train)

traj_transformations = []
if ENABLE_ROTATION_AUG: traj_transformations.append(random_rotation_wrapper)
if ENABLE_SCALING_AUG: traj_transformations.append(random_scaling_wrapper)
# Create dataloaders and applies the augmentation for training set only
training_data = TrajectoryDataset(train_objs_pos*size_modifier, train_traj_pos*size_modifier,
                                  n_dims, transform=traj_transformations, max_seq_len=max_len)
valid_data = TrajectoryDataset(valid_objs_pos, valid_traj_pos, n_dims, max_seq_len=max_len)

train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)
valid_dataloader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# # Training Demonstration Plot
# # %matplotlib notebook
# # shows random set of augmented trajectory
# # matplotlib.rcParams.update({'font.size': 10})
# fig = plt.figure(figsize = (8, 6))
# ax = fig.add_subplot(1, 1, 1, projection='3d')
# ax.set_facecolor('white')
# ax.locator_params(nbins=3, axis='z')
# colors = ['red', 'blue', 'yellow', 'orange', 'green', 'purple','pink']
#
# # def sim2real_array_op(lst):
# #     for array in lst:
# #         array[:, [2, 0]] = array[:, [0, 2]]
# #         array[:, 2] = -array[:, 2]
# #     return lst
#
# for i_data, sample_data in enumerate(training_data):
#
#     obj_seq, traj_seq, _, _, dm_idx = sample_data
#     if obj_seq[0,-3]!=1: continue
#     # sim2real_array_op([traj_seq, obj_seq])
#     traj_seq = traj_seq[traj_seq[:,n_dims]==1,:3].numpy() * train_std + train_mean
#     obj1_pos = obj_seq[1,:3] * train_std + train_mean
#     obj0_pos = obj_seq[0,:3] * train_std + train_mean
#     line = ax.plot(traj_seq[:,2], traj_seq[:,1], -traj_seq[:,0], '--', color=colors[i_data%len(colors)],
#                    label = f'demo {i_data}')
# #     traj_seq = traj_seq[traj_seq[:,n_dims]==1,:3].numpy() * train_std + train_mean
# #     obj1_pos = obj_seq[1,:3] * train_std + train_mean
# #     obj0_pos = obj_seq[0,:3] * train_std + train_mean
# #     line = ax.plot(traj_seq[:,0], traj_seq[:,1], traj_seq[:,2], '--', color=colors[i_data%len(colors)], label = f'demo {i_data}')
#     ax.plot(obj1_pos[2], obj1_pos[1], -obj1_pos[0], 'o',
#             color=colors[i_data%len(colors)], label=f'{i_data}')
#     ax.plot(obj0_pos[2], obj0_pos[1], -obj0_pos[0], 'x',
#             color=colors[i_data%len(colors)], label=f'{i_data}')
#
# ax.set_xlabel('x (mm)')
# ax.set_ylabel('y (mm)')
# ax.set_zlabel('z (mm)')
# # ax.set_box_aspect([ub - lb for lb, ub in (getattr(ax, f'get_{a}lim')() for a in 'xyz')])
# # handles, labels = ax.get_legend_handles_labels()
# # newHandles_temp, newLabels_temp = remove_repetitive_labels(handles, labels)
# # newLabels, newHandles = [], []
#
# # for handle, label in zip(newHandles_temp, newLabels_temp):
# #     if label not in ['start', 'middle', 'end']:
# #         newLabels.append(label)
# #         newHandles.append(handle)
# # plt.legend(newHandles, newLabels, loc = 'upper left',  prop={'size': 10})
# plt.show()

In [ ]:
# Training Demonstration Plot
# %matplotlib notebook
# shows random set of augmented trajectory
plt.rcParams.update({'font.size': 14})
fig = plt.figure(figsize = (6, 6))
ax = fig.add_subplot(1, 1, 1, projection='3d')
ax.set_facecolor('white')
ax.locator_params(nbins=3, axis='z')
colors = ['red', 'blue', 'yellow', 'orange', 'green', 'purple','pink']


plot_augmentation_type = 'rotation'
random.seed(103)
for i_data, sample_data in enumerate(training_data):

    obj_seq, traj_seq, _, _, dm_idx = sample_data
    if obj_seq[0,-3]!=1 or dm_idx!=31: continue

    # sim2real_array_op([traj_seq, obj_seq])
    traj_seq = traj_seq[traj_seq[:,n_dims]==1,:3].numpy() * train_std + train_mean
    obj_seq = obj_seq[:,:3].numpy() * train_std + train_mean
    obj1_pos = obj_seq[1,:3]
    obj0_pos = obj_seq[0,:3]
    line = ax.plot(traj_seq[:,2], traj_seq[:,1], -traj_seq[:,0], '--', color=colors[i_data%len(colors)])

    if plot_augmentation_type=="scaling": transformed_obj_pos, transformed_traj = random_scaling_wrapper(obj_seq, traj_seq, 0.25)
    elif plot_augmentation_type=="rotation": transformed_obj_pos, transformed_traj = random_rotation_wrapper(obj_seq, traj_seq)
    line = ax.plot(transformed_traj[:,2], transformed_traj[:,1], -transformed_traj[:,0], '--', color=colors[(i_data+1)%len(colors)])
#     traj_seq = traj_seq[traj_seq[:,n_dims]==1,:3].numpy() * train_std + train_mean
#     obj1_pos = obj_seq[1,:3] * train_std + train_mean
#     obj0_pos = obj_seq[0,:3] * train_std + train_mean
#     line = ax.plot(traj_seq[:,0], traj_seq[:,1], traj_seq[:,2], '--', color=colors[i_data%len(colors)], label = f'demo {i_data}')
    center_length= 30
    ax.plot(obj1_pos[2], obj1_pos[1], -obj1_pos[0], 'o',
            color=colors[i_data%len(colors)], label=f'old cup center')
    ax.plot(obj0_pos[2], obj0_pos[1], -obj0_pos[0], 'x',
            color=colors[i_data%len(colors)], label=f'old pitcher center')
    ax.plot(transformed_obj_pos[1,2], transformed_obj_pos[1,1], -transformed_obj_pos[1,0], 'o',
            color=colors[(i_data+1)%len(colors)], label=f'new cup center')
    ax.plot(transformed_obj_pos[0,2], transformed_obj_pos[0,1], -transformed_obj_pos[0,0], 'x',
            color=colors[(i_data+1)%len(colors)], label=f'new pitcher center')

ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_zlabel('z (mm)')
plt.legend(loc = 'upper left',  prop={'size': 12})
plt.savefig(os.path.join(GRAPH_DIR_PATH, f'traj_{plot_augmentation_type}.svg'), format='svg', dpi=300, transparent=True)
plt.show()

In [ ]:
# Load trained model
test_data = TrajectoryDataset(test_objs_pos, test_traj_pos, n_dims)
plot_dataloader = DataLoader(test_data, batch_size=1)
plot_colors = ['blue','black','green','orange','purple']

In [ ]:
torch_model = f"{MODEL_SUB_DIR}/TP-Tf_7D_{30}demos_3d{3}e_model{0}_{5000}epochs.pt"
load_model_file = os.path.join(MODEL_DIR_PATH, torch_model)
saved_file = torch.load(load_model_file)
saved_model = saved_file['model']

In [ ]:
# Run and save prediction result on the test set
inference_times = {i:[] for i in range(n_tasks)}
training_times = []
repeats = 5
for training_size in n_demos_list:
    for ith in range(MODEL_COPIES):
        saved_traj_dir = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/{running_model}_{DIR_SUFFIX}")
        if running_model== "TP-Tf":
            elayer = 3
            torch_model = f"{MODEL_SUB_DIR}/TP-Tf_7D_{training_size}demos_3d{elayer}e_model{ith}_{MAX_EPOCHS}epochs.pt"
            saved_traj_file = os.path.join(saved_traj_dir, f"saved_trajs_7d_{training_size}_3d{elayer}e_model{ith}.pkl")
        elif running_model== "LSTM_MSE":
            torch_model = f"{MODEL_SUB_DIR}/LSTM_MSE_7D_{training_size}demos_hdim48_model{ith}_{MAX_EPOCHS}epochs.pt"
            saved_traj_file = os.path.join(saved_traj_dir, f"saved_trajs_7d_{training_size}_model{ith}.pkl")

        pd_traj_pts = {}

        # Load LSTM and Transformer models
        load_model_file = os.path.join(MODEL_DIR_PATH, torch_model)
        try:
            saved_file = torch.load(load_model_file)
        except FileNotFoundError:
            print(f"Could not load: {load_model_file}")
            continue

        print(f"Loaded successfully: {load_model_file}")
        saved_model = saved_file['model']
        saved_model.eval()
        print("model size", get_n_params(saved_model))
        training_times.append(saved_file['time_passed'])
        for demo_input in plot_dataloader:
            obj_seq, traj_seq, traj_target, traj_weights, demo_idx= demo_input
            mask = traj_seq[0,:,n_dims]==1
            task_i = find_task_by_index(demo_idx, test_splits)
            # Check belong to task
            obj_seq_input = obj_seq.to(device)
            traj_target_input = traj_target.to(device)

            if saved_model.__class__.__name__=="TFEncoderDecoder":
                start = time.time()
                predicted_traj_tf = saved_model(obj_seq_input, traj_target_input).cpu()
                inference_times[task_i].append(time.time() - start)
                predicted_traj = predicted_traj_tf.detach().numpy()[0,mask]
            if saved_model.__class__.__name__=="TFEncoderDecoderCompact":
                start = time.time()
                predicted_traj_tf = saved_model(obj_seq_input, traj_target_input).cpu()
                inference_times[task_i].append(time.time() - start)
                predicted_traj = predicted_traj_tf.detach().numpy()[0,mask]
            if saved_model.__class__.__name__=="LSTM_MSE":
                saved_model.seq_len = mask.sum().detach().numpy()
                start = time.time()
                predicted_traj_tf = saved_model(obj_seq_input, obj_seq_input[:,0,-n_tasks:]).cpu()
                inference_times[task_i].append(time.time() - start)
                predicted_traj = predicted_traj_tf.detach().numpy()[0]


            # additional test-time augmentation
            sampled_pred = []
            for _ in  range(repeats):
                obj_seq_arr = obj_seq_input.cpu().numpy()
                deg, rot_center = random.randrange(0, 360), np.mean(obj_seq_arr[0,:,:3], axis=0) + np.random.normal(size=3) * np.std(obj_seq_arr)
            
                obj_seq_input_rotated = apply_rotate(obj_seq_arr[0,:,:7], deg, rot_center, axis='x')
                obj_seq_input[:,:,:7] = torch.tensor(obj_seq_input_rotated)
                output_seq = saved_model(obj_seq_input, traj_target_input).cpu().detach()[0][mask,:]
                output_seq = apply_rotate(output_seq.numpy(), -deg, rot_center, axis='x')
                sampled_pred.append(output_seq)
            predicted_traj = np.mean(sampled_pred, axis=0)


            predicted_traj[:,:3] = predicted_traj[:,:3]*train_std + train_mean
            pd_traj_pts[all_test_demos[demo_idx]] = predicted_traj
        if not os.path.exists(saved_traj_dir): os.mkdir(saved_traj_dir)
        with open(saved_traj_file, "wb") as fout:
            pickle.dump(pd_traj_pts, fout)
#
# for k, v in inference_times.items():
#     print(f"{running_model} Average Inference Time: {mean(v)}s for Task {task_names[k]}")
# print(f"Training Time: {mean(training_times)/3600} hours")

In [ ]:
# plt.rcParams.update({'font.size': 14})
# fig = plt.figure(figsize = (6, 6))
# ax = fig.add_subplot(1, 1, 1, projection='3d')
# ax.set_facecolor('white')
# ax.locator_params(nbins=3, axis='z')
#
# for demo_input in plot_dataloader:
#     obj_seq, traj_seq, traj_target, traj_weights, demo_idx= demo_input
#     mask = traj_seq[0,:,n_dims]==1
#     task_i = find_task_by_index(demo_idx, test_splits)
#     # Check belong to task
#     obj_seq_input = obj_seq.to(device)
#     traj_target_input = traj_target.to(device)
#     # additional test-time augmentation
#     sampled_pred = []
#     for _ in  range(repeats):
#         obj_seq_arr = obj_seq_input.cpu().numpy()
#         deg, rot_center = random.randrange(0, 360), np.mean(obj_seq_arr[0,:,:3], axis=0) + np.random.normal(size=3) * np.std(obj_seq_arr)
#
#         obj_seq_input_rotated = apply_rotate(obj_seq_arr[0,:,:7], deg, rot_center, axis='x')
#         obj_seq_input[0,:,:7] = torch.tensor(obj_seq_input_rotated)
#         output_seq = saved_model(obj_seq_input, traj_target_input).cpu().detach()[0][mask,:]
#         output_seq = apply_rotate(output_seq.numpy(), -deg, rot_center, axis='x')
#         sampled_pred.append(output_seq)
#     predicted_traj = np.mean(sampled_pred, axis=0)
#     print(obj_seq_arr.shape, output_seq.shape, traj_seq[0,mask].shape)
#     line = ax.plot(obj_seq_arr[0,:,2], obj_seq_arr[0,:,1], -obj_seq_arr[0,:,0], '--', color='green')
#     line = ax.plot(obj_seq_input_rotated[:,2], obj_seq_input_rotated[:,1], -obj_seq_input_rotated[:,0], '--', color='orange')
#     line = ax.plot(output_seq[:,2], output_seq[:,1], -output_seq[:,0], '--', color='blue')
#     line = ax.plot(traj_seq[0,mask,2], traj_seq[0,mask,1], -traj_seq[0,mask,0],  color="red")
#     break
#
# ax.set_xlabel('x (mm)')
# ax.set_ylabel('y (mm)')
# ax.set_zlabel('z (mm)')
# plt.show()

In [ ]:
# Check only 3d3e transformer currently
# for i, n_demos in enumerate([30]):
#     model_traj_datasets = []
#     saved_traj_file = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/TP-Tf_{DIR_SUFFIX}", f"saved_trajs_7d_{n_demos}_3d3e_model{kth}.pkl")
#     with open(saved_traj_file, "rb") as fout: model_traj_datasets.append(pickle.load(fout))
#
#     # saved_traj_file = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/LSTM_MSE_{DIR_SUFFIX}", f"saved_trajs_7d_{n_demos}_model{kth}.pkl")
#     # with open(saved_traj_file, "rb") as fout: model_traj_datasets.append(pickle.load(fout))
#
#     all_models = [f"{sm}-{kth}" for sm in main_models] + [f"{sm}-{kth}" for sm in secondary_models]
#     #  ==== Load LSTM and Transformer model predictions ====
#     for mdl_idx, traj_dataset in enumerate(model_traj_datasets):
#         predicted_traj = traj_dataset[all_test_demos[demo_idx]]
#         d, q = summarize_traj(predicted_traj, scaled_traj_seq.numpy())
#         line = ax2.plot(predicted_traj[:,2], predicted_traj[:,1], -predicted_traj[:,0], '-',
#                         label=f'{main_models[mdl_idx].upper()}', color=plot_colors[mdl_idx])
#
#     # Get recorded trajectory for secondary models
#     saved_traj_file = os.path.join(LOG_DIR, SAVED_TRAJ_VER_SECONDARY, f"saved_trajs_7d_{n_demos}_{tfile}_model0.pkl")
#     with open(saved_traj_file, "rb") as fout:
#         model_outputs = pickle.load(fout)
#         for m, md in enumerate(secondary_models[:2]):
#             if md not in secondary_models: continue
#             task_model_traj = model_outputs[md][task_demo_idx]
#             line2 = ax2.plot(task_model_traj[:,2], task_model_traj[:,1], -task_model_traj[:,0],
#                              '--', color=plot_colors[len(main_models) + m], label = f'{md.upper()}')
#             d = get_position_difference_per_step((task_model_traj[:,:3]-train_mean)/train_std, traj_seq.numpy()[0,mask,:3])*train_std
#             q = get_quaternion_difference_per_step(task_model_traj[:,3:7], traj_seq.numpy()[0,mask,3:7])
#     mat_traj_file = os.path.join(LOG_DIR, "tests", task_old_names[task_idx], "result", f"kmp_predictions_{n_demos}demos_{0}.mat")
#     kmp_outputs = scipy.io.loadmat(mat_traj_file)['kmp_predictions']
#     t_idx = task_demo_idx[0].item()
#     task_model_traj = kmp_outputs[task_demo_idx]
#     line2 = ax2.plot(task_model_traj[:,2], task_model_traj[:,1], -task_model_traj[:,0], '--', color=plot_colors[4], label = f'KMP')

In [ ]:
# # Select demo from data set based on index and plot predicted trajectory
# plt.close('all')
# plt.rcParams.update({'font.size':18})
# fig = plt.figure(figsize = (8, 8))
# ax2 = fig.add_subplot(1, 1, 1, projection='3d')
#
# target_index = 27
# iterable_loader = iter(plot_dataloader)
# demo_input = next(iterable_loader)
# while True:
#     demo_input = next(iterable_loader)
#     obj_seq, traj_seq, traj_target, _, demo_idx = demo_input
#     if target_index==demo_idx: break
#
# obj_seq_input = obj_seq.to(device)
# # traj_target_input = traj_target.to(device)
# # target_key_mask = traj_target.clone().sum(dim=2).le(1).to(device)
#
# task_idx = find_task_by_index(demo_idx, test_splits)
# tfile = task_files[task_idx]
# task_demo_idx = demo_idx - test_splits[task_idx]
#
# obj0_pos = obj_seq[0,0,:3]*train_std+train_mean
# obj1_pos = obj_seq[0,1,:3]*train_std+train_mean
#
# mask = traj_seq[0,:,n_dims]==1
# scaled_traj_seq = traj_seq[0,mask]
# scaled_traj_seq[:,:3] = scaled_traj_seq[:,:3]*train_std+train_mean
# line = ax2.plot(scaled_traj_seq[:,2].tolist(), scaled_traj_seq[:,1].tolist(), (-scaled_traj_seq[:,0]).tolist(),
#                  color='red', label = f'ground truth')
# ax2.plot(obj0_pos[2], obj0_pos[1], -obj0_pos[0], 'o',
#         color='red', label=f'{unique_objs[to_obj_index(obj_seq[0,0,][n_dims:n_dims+n_objs])]}')
# ax2.plot(obj1_pos[2], obj1_pos[1], -obj1_pos[0], 'x',
#         color='red', label=f'{unique_objs[to_obj_index(obj_seq[0,1,][n_dims:n_dims+n_objs])]}')
#
# # Check only 3d3e transformer currently
# for i, n_demos in enumerate([30]):
#     model_traj_datasets = []
#     saved_traj_file = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/TP-Tf_{DIR_SUFFIX}", f"saved_trajs_7d_{n_demos}_3d3e_model{kth}.pkl")
#     with open(saved_traj_file, "rb") as fout: model_traj_datasets.append(pickle.load(fout))
#
#     saved_traj_file = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/LSTM_MSE_{DIR_SUFFIX}", f"saved_trajs_7d_{n_demos}_model{kth}.pkl")
#     with open(saved_traj_file, "rb") as fout: model_traj_datasets.append(pickle.load(fout))
#
#     all_models = [f"{sm}-{kth}" for sm in main_models] + [f"{sm}-{kth}" for sm in secondary_models]
#     #  ==== Load LSTM and Transformer model predictions ====
#     for mdl_idx, traj_dataset in enumerate(model_traj_datasets):
#         predicted_traj = traj_dataset[all_test_demos[demo_idx]]
#         d, q = summarize_traj(predicted_traj, scaled_traj_seq.numpy())
#         line = ax2.plot(predicted_traj[:,2], predicted_traj[:,1], -predicted_traj[:,0], '-',
#                         label=f'{main_models[mdl_idx].upper()}', color=plot_colors[mdl_idx])
#
#     saved_traj_file = os.path.join(LOG_DIR, SAVED_TRAJ_VER_SECONDARY, f"saved_trajs_7d_{n_demos}_{tfile}_model0.pkl")
#     with open(saved_traj_file, "rb") as fout:
#         model_outputs = pickle.load(fout)
#         for m, md in enumerate(secondary_models[:2]):
#             if md not in secondary_models: continue
#             task_model_traj = model_outputs[md][task_demo_idx]
#             line2 = ax2.plot(task_model_traj[:,2], task_model_traj[:,1], -task_model_traj[:,0],
#                              '--', color=plot_colors[len(main_models) + m], label = f'{md.upper()}')
#             d = get_position_difference_per_step((task_model_traj[:,:3]-train_mean)/train_std, traj_seq.numpy()[0,mask,:3])*train_std
#             q = get_quaternion_difference_per_step(task_model_traj[:,3:7], traj_seq.numpy()[0,mask,3:7])
#     mat_traj_file = os.path.join(LOG_DIR, "tests", task_old_names[task_idx], "result", f"kmp_predictions_{n_demos}demos_{0}.mat")
#     kmp_outputs = scipy.io.loadmat(mat_traj_file)['kmp_predictions']
#     t_idx = task_demo_idx[0].item()
#     task_model_traj = kmp_outputs[task_demo_idx]
#     line2 = ax2.plot(task_model_traj[:,2], task_model_traj[:,1], -task_model_traj[:,0], '--', color=plot_colors[4], label = f'KMP')
#
# ax2.set_xlabel('x (mm)', labelpad=16)
# ax2.set_ylabel('y (mm)', labelpad=16)
# ax2.set_zlabel('z (mm)', labelpad=16)
# ax2.locator_params(axis='both', nbins=3)
# ax2.set_box_aspect([ub - lb for lb, ub in (getattr(ax2, f'get_{a}lim')() for a in 'xyz')])
# # ax2.view_init(40, -80, 0)
# plt.legend(bbox_to_anchor=(.6, .66), frameon=False)
# # plt.savefig(f'./graphs/demo{target_index}.svg', format='svg', transparent=True)
# plt.show()


In [ ]:
# Get all model averages across task and size
# Nesting order: 1.Demos size, 2.Task index, 3.Demo index, 4.Models,
pd_traj_pts, saved_pos_avgs, saved_quat_avgs, saved_chamfer_avgs = {}, {}, {}, {}
agg_result_df = pd.DataFrame(columns=['demo_size', 'task_id', 'model', 'ADE', 'NDQ'])
for n_demos in n_demos_list:
    pd_traj_pts[n_demos], saved_pos_avgs[n_demos], saved_quat_avgs[n_demos], saved_chamfer_avgs[n_demos] = {}, {}, {}, {}
    print("\n",'='*8, n_demos, "Unique Training Demos",'='*8)
    for i, file in enumerate(selected_tasks):
        pd_traj_pts[n_demos][i], saved_pos_avgs[n_demos][i], saved_quat_avgs[n_demos][i], saved_chamfer_avgs[n_demos][i] = {}, {}, {}, {}
        print(f"{selected_tasks[i]} task",'-'*68)
        models_dist, models_norm_diff, model_chamfer = {}, {}, {}
        for k in range(MODEL_COPIES):
            model_traj_datasets = []
            if only_ith_task >= 0: traj_file = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/{running_model}_{DIR_SUFFIX}", f"saved_trajs_7d_{n_demos}_3d3e_model{k}.pkl")
            else: traj_file = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/TP-Tf_{DIR_SUFFIX}", f"saved_trajs_7d_{n_demos}_3d3e_model{k}.pkl")
            if os.path.exists(traj_file): model_traj_datasets.append(pickle.load(open(traj_file, "rb")))
            traj_file = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/LSTM_MSE_{DIR_SUFFIX}", f"saved_trajs_7d_{n_demos}_model{k}.pkl")
            if os.path.exists(traj_file): model_traj_datasets.append(pickle.load(open(traj_file, "rb")))
            all_models = [f"{m}+{k}" for m in main_models] + [f"{m}+{k}" for m in secondary_models]
            for mdl_name_k in all_models:
                models_dist[mdl_name_k] = []
                models_norm_diff[mdl_name_k] = []
                model_chamfer[mdl_name_k] = []
                pd_traj_pts[n_demos][i][mdl_name_k] = {}
            for demo_input in plot_dataloader:
                obj_seq, traj_seq, traj_target, _, demo_idx = demo_input
                mask = traj_seq[0,:,n_dims]==1
                traj_seq[0,:,:3] = traj_seq[0,:,:3]*train_std + train_mean
                # Check belong to task
                if to_obj_index(task_tags[file])!=to_obj_index(obj_seq[0,0,-n_tasks:]): continue
                obj_seq_input = obj_seq.to(device)
                traj_target_input = traj_target.to(device)
                cup_pos = obj_seq[0,end_obj_index[i],:3].numpy()*train_std + train_mean

                #  ==== Load LSTM and Transformer model predictions ====
                for tm_idx, traj_dataset in enumerate(model_traj_datasets):
                    model_name = f"{main_models[tm_idx]}+{k}"
                    predicted_traj = traj_dataset[all_test_demos[demo_idx]]
                    d, q = summarize_traj(predicted_traj, traj_seq.numpy()[0,mask])
                    chamfer = chamfer_dist(predicted_traj[:, :3], traj_seq.numpy()[0,mask, :3])
                    pd_traj_pts[n_demos][i][model_name][all_test_demos[demo_idx]] = predicted_traj[:,:3]
                    models_dist[model_name].append([d[0], d[-1], np.mean(d)])
                    models_norm_diff[model_name].append([q[0], q[-1], np.mean(q)])
                    model_chamfer[model_name].append(chamfer)
                    task_idx = find_task_by_index(demo_idx, test_splits)
                    tfile = task_files[task_idx]
                    task_demo_idx = demo_idx - test_splits[task_idx]

                # ==== Load Movement Primitives and GMM model predictions ====
                secondary_traj_file = os.path.join(LOG_DIR, SAVED_TRAJ_VER_SECONDARY, f"saved_trajs_7d_{n_demos}_{file}_model{k}.pkl")
                if os.path.exists(secondary_traj_file):
                    model_outputs = pickle.load(open(secondary_traj_file, "rb"))
                    for m, mdl in enumerate(model_outputs.keys()):
                        total_test_size = len(model_outputs[mdl])
                        if total_test_size < 1: continue
                        t_idx = task_demo_idx[0].item()
                        task_model_traj = model_outputs[mdl][t_idx]
                        d, q = summarize_traj(task_model_traj, traj_seq.numpy()[0,mask])
                        chamfer = chamfer_dist(task_model_traj[:, :3], traj_seq.numpy()[0,mask, :3])
                        model_name = f"{mdl}+{k}"
                        models_dist[model_name].append([d[0], d[-1], np.mean(d)])
                        models_norm_diff[model_name].append([q[0], q[-1], np.mean(q)])
                        model_chamfer[model_name].append(chamfer)
                        pd_traj_pts[n_demos][i][model_name][all_test_demos[demo_idx]] = task_model_traj[:,:3]

                # ==== Load separate KMP model predictions ====
                kmp_traj_file = os.path.join(LOG_DIR, "tests", task_old_names[i], "result", f"kmp_predictions_{n_demos}demos_{k}.mat")
                if os.path.exists(kmp_traj_file):
                    kmp_outputs = scipy.io.loadmat(kmp_traj_file)['kmp_predictions']
                    t_idx = task_demo_idx[0].item()
                    task_model_traj = kmp_outputs[t_idx]
                    d, q = summarize_traj(task_model_traj, traj_seq.numpy()[0,mask])
                    chamfer = chamfer_dist(task_model_traj[:, :3], traj_seq.numpy()[0,mask, :3])
                    model_name = f"kmp+{k}"
                    models_dist[model_name].append([d[0], d[-1], np.mean(d)])
                    models_norm_diff[model_name].append([q[0], q[-1], np.mean(q)])
                    model_chamfer[model_name].append(chamfer)
                    pd_traj_pts[n_demos][i][model_name][all_test_demos[demo_idx]] = task_model_traj[:,:3]
        for model_name in main_models + secondary_models:
            combined_avg = []
            for k in range(MODEL_COPIES):
                model_k = f"{model_name}+{k}"
                if model_k not in models_dist or len(models_dist[model_k]) < 1: continue
                tp_mean_ds = np.mean(np.array(models_dist[model_k]), axis = 0)
                tp_mean_qs = np.mean(np.array(models_norm_diff[model_k]), axis = 0)
                tp_mean_cham = np.mean(model_chamfer[model_k])
                agg_result_df.loc[len(agg_result_df.index)] = [n_demos, task_names[i], model_k.split('+')[0], tp_mean_ds[-1], tp_mean_qs[-1]]
                print(f"{model_k}({n_dims}D):\nPos Diff - S: {round(tp_mean_ds[0],2)}, E: {round(tp_mean_ds[1],2)}, A: {round(tp_mean_ds[-1],2)}mm",
                      f"Quat - S: {round(tp_mean_qs[0],3)}, E: {round(tp_mean_qs[1],3)}, A: {round(tp_mean_qs[-1],3)}")
                # Save the average
                saved_pos_avgs[n_demos][i][model_k] = tp_mean_ds[-1]
                saved_quat_avgs[n_demos][i][model_k] = tp_mean_qs[-1]
                saved_chamfer_avgs[n_demos][i][model_k] = tp_mean_cham
                combined_avg.append(tp_mean_ds[-1])
                if k==MODEL_COPIES-1: print(f"Averages for model {model_name}: {mean(combined_avg)}")

In [ ]:
# # print result for differences in encoder layers and augmentations for TP-Transformer
# encoder_layers = [0, 1, 2, 3]
# aug_grid = [g.ravel() for g in np.meshgrid([0,1], [0,1])]
# agg_result_df = pd.DataFrame(columns=['demo_size', 'task_id', 'model', 'encoder_layers', 'rotation_enabled', 'scaling_enabled', 'ADE', 'NDQ'])
# saved_pos_avgs, saved_quat_avgs = {}, {}
# for elayers in encoder_layers:
#     for j in range(4):
#         enable_rotation, enable_scaling = aug_grid[0][j], aug_grid[1][j]
#         for n_demos in [30]:
#             saved_pos_avgs[n_demos], saved_quat_avgs[n_demos]= {}, {}
#             for i, file in enumerate(task_files):
#                 saved_pos_avgs[n_demos][i], saved_quat_avgs[n_demos][i] = {}, {}
#                 print(file,'-'*68)
#                 models_dist, models_norm_diff = {}, {}
#                 for k in range(MODEL_COPIES):
#                     model_traj_datasets = []
#
#                     try:
#                         traj_file = os.path.join(LOG_DIR, f"{TRAJ_SAVE_DIR}/TP-Tf_rot{1 if enable_rotation else 0}_scale{1 if enable_scaling else 0}", f"saved_trajs_7d_{n_demos}_3d{elayers}e_model{k}.pkl")
#                         with open(traj_file, "rb") as fout: model_traj_datasets.append(pickle.load(fout))
#                     except:
#                         continue
#                     print(f"Loaded File: {traj_file}")
#                     all_models = [f"{sm}+{k}" for sm in main_models] + [f"{sm}+{k}" for sm in secondary_models]
#                     for mdl_name_k in all_models:
#                         models_dist[mdl_name_k] = []
#                         models_norm_diff[mdl_name_k] = []
#                     for demo_input in plot_dataloader:
#                         obj_seq, traj_seq, traj_target, _, demo_idx = demo_input
#                         mask = traj_seq[0,:,n_dims]==1
#                         traj_seq[0,:,:3] = traj_seq[0,:,:3]*train_std + train_mean
#                         # Check belong to task
#                         if to_obj_index(task_tags[file])!=to_obj_index(obj_seq[0,0,-n_tasks:]): continue
#                         obj_seq_input = obj_seq.to(device)
#                         traj_target_input = traj_target.to(device)
#                         cup_pos = obj_seq[0,end_obj_index[i],:3].numpy()*train_std + train_mean
#
#                         #  ==== Load LSTM and Transformer model predictions ====
#                         for tm_idx, traj_dataset in enumerate(model_traj_datasets):
#                             model_name = f"{main_models[tm_idx]}+{k}"
#                             predicted_traj = traj_dataset[all_test_demos[demo_idx]]
#                             d, q = summarize_traj(predicted_traj, traj_seq.numpy()[0,mask])
#                             models_dist[model_name].append([d[0], d[-1], np.mean(d)])
#                             models_norm_diff[model_name].append([q[0], q[-1], np.mean(q)])
#                             task_idx = find_task_by_index(demo_idx, test_splits)
#                             tfile = task_files[task_idx]
#                             task_demo_idx = demo_idx - test_splits[task_idx]
#
#                 for model_k in models_dist.keys():
#                     if len(models_dist[model_k]) < 1: continue
#                     tp_mean_ds = np.mean(np.array(models_dist[model_k]), axis = 0)
#                     tp_mean_qs = np.mean(np.array(models_norm_diff[model_k]), axis = 0)
#                     agg_result_df.loc[len(agg_result_df.index)] = [n_demos, task_names[i], model_k.split('+')[0], elayers, enable_rotation, enable_scaling, tp_mean_ds[-1], tp_mean_qs[-1]]
#                     print(f"{model_k}({n_dims}D):\nPos Diff - S: {round(tp_mean_ds[0],2)}, E: {round(tp_mean_ds[1],2)}, A: {round(tp_mean_ds[-1],2)}mm",
#                           f"Quat - S: {round(tp_mean_qs[0],3)}, E: {round(tp_mean_qs[1],3)}, A: {round(tp_mean_qs[-1],3)}")
#                     # Save the average
#                     saved_pos_avgs[n_demos][i][model_k] = tp_mean_ds[-1]
#                     saved_quat_avgs[n_demos][i][model_k] = tp_mean_qs[-1]

In [ ]:
# agg_means = agg_result_df.groupby(["demo_size", "task_id", "model", 'encoder_layers', 'rotation_enabled', 'scaling_enabled']).mean()
# agg_means.rename(columns={"ADE":"ADE_Mean", "NDQ":"NQD_Mean"}, inplace=True)
# agg_stds = agg_result_df.groupby(["demo_size", "task_id", "model", 'encoder_layers', 'rotation_enabled', 'scaling_enabled']).std()
# agg_stds.rename(columns={"ADE":"ADE_STD", "NDQ":"NQD_STD"}, inplace=True)
# agg_table = pd.concat([agg_means, agg_stds], axis="columns")
# moved_column = agg_table.pop("ADE_STD")
# agg_table.insert(1, 'ADE_STD', moved_column)
# agg_table = agg_table.round(3)
# agg_table

In [ ]:
# Model average and standard deviation
plt.rcParams.update({'font.size': 14})
fig = plt.figure(figsize = (16, 5))
fig.tight_layout(pad=2.0)
axs = fig.subplots(1, len(task_files))
lims = (-600, 600)
model_line_types = ['-','-','--','--','--']
PLOT_MEASURE = "ADE"

if PLOT_MEASURE=="ADE":
    saved_avgs, upper_lim = saved_pos_avgs, 140
    ylabel_name, plot_filename = "ADE (mm)", 'models_position_error.svg'
# elif PLOT_MEASURE=="chamfer":
#     saved_avgs, upper_lim = saved_chamfer_avgs, 140
#     ylabel_name, plot_filename = "Chamfer Distance (mm)", 'models_chamfer_error'
elif PLOT_MEASURE=="NDQ":
    saved_avgs, upper_lim = saved_quat_avgs, 0.5
    ylabel_name, plot_filename = "NDQ", 'models_quaternion_error.svg'
else:
    raise NameError("No such measurement exists")
all_models = ["TP-Tf", "LSTM_MSE", "tp-gmm", "tp-pmp", "kmp"]
for i, file in enumerate(task_files):
    model_avg_mean, model_avg_std = {m:[] for m in all_models}, {m:[] for m in all_models}
    n_demos_available = [n for n in n_demos_list if n in saved_avgs.keys()]
    for n_demos in n_demos_available:
        temp_avgs = {m:[] for m in all_models}
        for k in range(MODEL_COPIES):
            for mdl in all_models:
                model_name = f"{mdl}+{k}"
                if model_name not in saved_avgs[n_demos][i].keys(): continue
                temp_avgs[mdl].append(saved_avgs[n_demos][i][model_name])
        for mdl in temp_avgs.keys():
            model_avg_mean[mdl].append(np.mean(temp_avgs[mdl]))
            model_avg_std[mdl].append(np.std(temp_avgs[mdl]))
    for m, mdl in enumerate(model_avg_mean.keys()):

        model_avg_mean[mdl] = np.array(model_avg_mean[mdl])
        model_avg_std[mdl] = np.array(model_avg_std[mdl])
        axs[i].plot(n_demos_available,  model_avg_mean[mdl], model_line_types[m], c=plot_colors[m], label=mdl.upper())
        axs[i].scatter(x = n_demos_available, y = model_avg_mean[mdl], s=8, c=plot_colors[m])
        axs[i].fill_between(n_demos_available, (model_avg_mean[mdl]-model_avg_std[mdl]),
                            (model_avg_mean[mdl]+model_avg_std[mdl]), color=plot_colors[m], alpha=.2)

    if i==0: axs[i].set_ylabel(ylabel_name)
    axs[i].set_xlabel(f"Demonstrations")
    axs[i].set_xlim(5, 30)
    axs[i].set_ylim(0, upper_lim)
    axs[i].set_xticks([5,10,20,30])
    axs[i].set_title(f"{task_names[i]}")

plt.legend(frameon=False)
plt.savefig(os.path.join(GRAPH_DIR_PATH, plot_filename), format='svg', dpi=300, bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
# Compare error rates
for i, file in enumerate(task_files):
    model_avg_mean, model_avg_std = {m:[] for m in all_models}, {m:[] for m in all_models}
    n_demos_available = [n for n in n_demos_list if n in saved_avgs.keys()]
    for n_demos in n_demos_available:
        temp_avgs = {m:[] for m in all_models}
        for k in range(MODEL_COPIES):
            for mdl in all_models:
                model_name = f"{mdl}+{k}"
                if model_name not in saved_avgs[n_demos][i].keys(): continue
                temp_avgs[mdl].append(saved_avgs[n_demos][i][model_name])
        for mdl in temp_avgs.keys():
            model_avg_mean[mdl].append(np.mean(temp_avgs[mdl]))
            model_avg_std[mdl].append(np.std(temp_avgs[mdl]))
    df = pd.DataFrame(model_avg_mean)
    for col in df.columns:
        if col=="TP-Tf" or col=="LSTM_MSE": continue
        df[f"{col}-err-diff"] = 1 - (df['TP-Tf']/df[col])
    print(i, df.iloc[:,5:])

In [ ]:
# ANOVA model performance comparison
import statsmodels.api as sm
from statsmodels.formula.api import ols
TEST_RESPONSE = 'ADE'
# f'{TEST_RESPONSE} ~ demo_size + task_id + model + task_id:model'
# Ordinary Least Squares (OLS) model
ols_model = ols(f'{TEST_RESPONSE} ~ demo_size + task_id + model + task_id:model + demo_size:model', data=agg_result_df).fit()
anova_table = sm.stats.anova_lm(ols_model, typ=3)
anova_table

In [ ]:
ols_model.summary()

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_df = pd.DataFrame(columns=[])
task_size = agg_result_df.task_id.unique().size
for task_k in range(task_size):
    selected_df = agg_result_df.where((agg_result_df.demo_size==30) & (agg_result_df.task_id==task_names[task_k])).dropna()
    # Perform Tukey's HSD test
    tukey = pairwise_tukeyhsd(endog=selected_df[TEST_RESPONSE], groups=selected_df['model'], alpha=0.05)
    tukey_df[f'reject_task#{task_k}'] = tukey.reject
    print(task_k, tukey)
print(tukey_df)

In [ ]:
#Check model performance vs epochs
# Plot performance vs epochs
plt.rcParams.update({'font.size': 14})
fig = plt.figure(figsize = (6, 5))
fig.tight_layout(pad=2.0)
axs = fig.subplots(1, 1)
epoch_plot_colors = ['red', 'blue', 'green', 'orange']
line_types = ['-', '--']
window_size = 9
kernel = np.ones(window_size) / window_size

loss_avg_epochs = {}
for i, file in enumerate(task_files):
    loss_avg_epochs[i] = {}
    for m, mdl in enumerate(main_models):
        loss_avg_epochs[i][mdl] = []
        for k in range(MODEL_COPIES):
            model_dir = f"{mdl}_{DIR_SUFFIX}"
            if mdl=="TP-Tf":
                torch_model = f"{model_dir}/TP-Tf_7D_30demos_3d3e_model{k}_{MAX_EPOCHS}epochs.pt"
            elif mdl=="LSTM_MSE":
                torch_model = f"{model_dir}/LSTM_MSE_7D_30demos_hdim200_model{k}_{MAX_EPOCHS}epochs.pt"
            load_model_file = os.path.join(MODEL_DIR_PATH, torch_model)
            saved_file = torch.load(load_model_file)
            saved_model_losses = saved_file['losses']
            loss_avg_epochs[i][mdl].append(saved_model_losses)

        vloss_per10epoch = np.array(loss_avg_epochs[i][mdl]).mean(axis=0)[:,1]
        # Smooth the data with the kernel
        data_convolved = np.convolve(vloss_per10epoch, kernel, mode='same')
        epoch_list = np.arange(0, vloss_per10epoch.shape[0]*10, 10)
        print(m, f"{task_names[i]} ({mdl.upper()})")
        axs.plot(epoch_list, data_convolved, line_types[m], c=epoch_plot_colors[i], label=f"{task_names[i]} ({mdl.upper()})")
axs.set_xlabel(f"Epochs")
axs.set_ylabel(f"Validation Loss (mm)")
axs.set_xlim(0, 20000)
axs.set_ylim(0, .3)
axs.set_title("Validation Loss versus Epoch")
plt.legend(bbox_to_anchor=(.35, .45), frameon=False)
plt.show()
plt.savefig(os.path.join(GRAPH_DIR_PATH, 'validation_vs_epoch.svg'), format='svg', dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
#Check model performance vs epochs
vdata = TrajectoryDataset(valid_objs_pos, valid_traj_pos, n_dims)
vdataloader = DataLoader(vdata, batch_size=1)
pd_epochs_avgs = {}
inference_avgs = {}
epoch_check_pts = list(range(0, 20001, 500))
for i, file in enumerate(task_files):
    pd_epochs_avgs[i] = {m:[] for m in main_models}
    inference_avgs[i] = {m:[] for m in main_models}
    for epoch_cpt in epoch_check_pts:
        models_dist, models_norm_diff = {m:[] for m in main_models}, {m:[] for m in main_models}
        for timed_mdl in main_models:
            MODEL_PATH_TIMED = os.path.join(MODEL_DIR_PATH, f'{timed_mdl}_{DIR_SUFFIX}_timed')
            if timed_mdl=='LSTM_MSE': model_name = f"{timed_mdl}_{n_dims}D_30demos_hdim200_model0_{epoch_cpt}epochs"
            elif timed_mdl=='TP-Tf': model_name = f"{timed_mdl}_{n_dims}D_30demos_3d3e_model0_{epoch_cpt}epochs"
            load_model_file = os.path.join(MODEL_PATH_TIMED, f"{model_name}.pt")
            saved_model = torch.load(load_model_file)["model"]
            saved_model.eval()
            for demo_input in vdataloader:
                obj_seq, traj_seq, traj_target, _, demo_idx = demo_input
                # Check belong to task
                if to_obj_index(task_tags[file])!=to_obj_index(obj_seq[0,0,-n_tasks:]): continue
                obj_seq_input = obj_seq.to(device)
                traj_target_input = traj_target.to(device)
                if saved_model.__class__.__name__=="TFEncoderDecoder":
                    mask = traj_seq[0,:,n_dims]==1
                    start_time = time.time()
                    predicted_traj = saved_model(obj_seq_input, traj_target_input).cpu().detach().numpy()[0,mask]
                    inference_avgs[i][timed_mdl].append(time.time() - start_time)
                elif saved_model.__class__.__name__=="LSTM_MSE":
                    saved_model.seq_len = mask.sum().detach().numpy()
                    start_time = time.time()
                    predicted_traj = saved_model(obj_seq_input, obj_seq_input[:,0,-n_tasks:]).cpu().detach().numpy()[0]
                    inference_avgs[i][timed_mdl].append(time.time() - start_time)
                d = get_position_difference_per_step(predicted_traj[:,:3], traj_seq.numpy()[0,mask,:3])*train_std
                q = get_quaternion_difference_per_step(predicted_traj[:,3:7], traj_seq.numpy()[0,mask,3:7])

                models_dist[timed_mdl].append([d[0], d[-1], np.mean(d)])
                models_norm_diff[timed_mdl].append([q[0], q[-1], np.mean(q)])

            tp_mean_ds = np.mean(np.array(models_dist[timed_mdl]), axis = 0)
            tp_mean_qs = np.mean(np.array(models_norm_diff[timed_mdl]), axis = 0)
            pd_epochs_avgs[i][timed_mdl].append([epoch_cpt, tp_mean_ds[-1]])
            if epoch_cpt==20000: print(file, timed_mdl, mean(inference_avgs[i][timed_mdl]))



In [ ]:
# Plot performance vs epochs
plt.close('all')
plt.rcParams.update({'font.size': 14})
fig = plt.figure(figsize = (6, 5))
fig.tight_layout(pad=2.0)
axs = fig.subplots(1, 1)

epoch_plot_colors = ['red', 'blue', 'green', 'orange']
line_types = ['-', '--']
window_size = 9
kernel = np.ones(window_size) / window_size
line_types = ['-', '--']
for i, _ in enumerate(task_files):
    for m, timed_mdl2 in enumerate(main_models):
        epoch_list, task_avgs = list(zip(*pd_epochs_avgs[i][timed_mdl2]))
        axs.plot(epoch_list,  task_avgs, line_types[m], c=epoch_plot_colors[i], label=f"{task_names[i]} ({timed_mdl2.upper()})")
        print(timed_mdl2, task_avgs[-1])
axs.set_xlabel(f"Epochs")
axs.set_ylabel(f"Prediction Error (mm)")
axs.set_xlim(0, 20000)
axs.set_ylim(0, 350)
axs.set_title("Validation Error versus Epoch")
plt.legend(loc='upper right', frameon=False)
plt.savefig(os.path.join(GRAPH_DIR_PATH, 'validation_vs_epoch.svg'), format='svg', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
TP-Tf 29.991766140341745
LSTM_MSE 26.58566091671721
TP-Tf 40.7655632133355
LSTM_MSE 45.9278300652189
TP-Tf 41.66673223108542
LSTM_MSE 57.299181403042496
TP-Tf 68.78586672308859
LSTM_MSE 101.11576218650356

In [ ]:
# Compare Encoder layers
model_tags = [0,1,2,3,'no_aug']
# model_tags = [0,1,2,3]

for _, file in enumerate(task_files):
    print('-'*70)
    models_dist = {t:[] for t in model_tags}
    models_norm_diff = {t:[] for t in model_tags}
    # Transformer models
    for demo_input in plot_dataloader:
        obj_seq, traj_seq, traj_target, _, demo_idx = demo_input
        # Check belong to task
        if to_obj_index(task_tags[file])!=to_obj_index(obj_seq[0,0,-n_tasks:]): continue
        obj_seq_input = obj_seq.to(device)
        traj_target_input = traj_target.to(device)

        for layers in range(5):
            for j in range(MODEL_COPIES):
                model_name = f"model_{n_dims}D_30demos_3d{layers}e_nvp_model{j}_{MAX_EPOCHS}epochs"
                model_name_version = model_name
                load_model_file = os.path.join(MODEL_PATH, f"{model_name_version}.pt")
                saved_file = torch.load(load_model_file)
                saved_model = saved_file['model']
                saved_model.eval()

                predicted_traj_tf = saved_model(obj_seq_input, traj_target_input).cpu()
                predicted_traj = predicted_traj_tf.detach().numpy()[0]
                mask = traj_seq[0,:,n_dims]==1
                d = get_position_difference_per_step(predicted_traj[mask,:3], traj_seq.numpy()[0,mask,:3])*train_std
                q = get_quaternion_difference_per_step(predicted_traj[mask,3:7], traj_seq.numpy()[0,mask,3:7])
                # Initialize
                models_dist[layers].append([d[0], d[-1], np.mean(d)])
                models_norm_diff[layers].append([q[0], q[-1], np.mean(q)])

        for j in range(MODEL_COPIES):
            model_name = f"model_{n_dims}D_30demos_3d3e_nvp_model{j}_{MAX_EPOCHS}epochs_noAug"
            model_name_version = model_name
            load_model_file = os.path.join(MODEL_PATH, f"{model_name_version}.pt")
            saved_file = torch.load(load_model_file)
            saved_model = saved_file['model']
            saved_model.eval()

            predicted_traj_tf = saved_model(obj_seq_input, traj_target_input).cpu()
            predicted_traj = predicted_traj_tf.detach().numpy()[0]
            mask = traj_seq[0,:,n_dims]==1
            d = get_position_difference_per_step(predicted_traj[mask,:3], traj_seq.numpy()[0,mask,:3])*train_std
            q = get_quaternion_difference_per_step(predicted_traj[mask,3:7], traj_seq.numpy()[0,mask,3:7])
            # Initialize
            models_dist['no_aug'].append([d[0], d[-1], np.mean(d)])
            models_norm_diff['no_aug'].append([q[0], q[-1], np.mean(q)])

    print(f"Task Name: {task_names[i]}")
    for model_k in models_dist.keys():
        tp_mean_ds = np.mean(np.array(models_dist[model_k]), axis = 0)
        tp_mean_qs = np.mean(np.array(models_norm_diff[model_k]), axis = 0)
        tp_std_ds = np.std(np.array(models_dist[model_k]), axis = 0)
        tp_std_qs = np.std(np.array(models_norm_diff[model_k]), axis = 0)
#             print(f"{task_names[i]} {model_k.upper()}({tp_model_dims}D):\nPositions Difference - Start: {round(tp_mean_ds[0],2)}, End: {round(tp_mean_ds[1],2)}mm, Average: {round(tp_mean_ds[-1],2)}mm")
#         print(f"{model_k}({n_dims}D):\nPos Diff - S: {round(tp_mean_ds[0],2)}, E: {round(tp_mean_ds[1],2)}, A: {round(tp_mean_ds[-1],2)}mm",
#               f"Quat - S: {round(tp_mean_qs[0],3)}, E: {round(tp_mean_qs[1],3)}, A: {round(tp_mean_qs[-1],3)}")
        print(f"{model_k}({n_dims}D):".ljust(20), f"Pos Diff - A: {round(tp_mean_ds[-1],2)}mm - S: {round(tp_std_ds[-1],2)}mm",
              f"Quat - A: {round(tp_mean_qs[-1],3)} - S: {round(tp_std_qs[-1],3)}mm")


# result_path = "./result"
# avg_file = os.path.join(result_path, f"avg_file{kth}.pkl")
# with open(avg_file, "wb") as fout:
#     pickle.dump(avg_dist_to_size, fout)

In [ ]:
test_data = TrajectoryDataset(test_objs_pos + valid_objs_pos, test_traj_pos + valid_traj_pos, n_dims)
plot_dataloader = DataLoader(test_data, batch_size=1)

#=================
obj_indice = [1, 1, 0, 0] # target objects indices
# obj_indice = [0, 0, 1, 1] # starting object indices
#=================

# Correlation Graph
plt.rcParams.update({'font.size': 18})
fig = plt.figure(figsize = (6, 15))
axs = fig.subplots(n_tasks, 1)
plt.subplots_adjust(left=0.1, right=0.95, top=0.975, bottom=0.05, wspace=0.4, hspace=0.3)
lims = (-600, 600)

plot_colors = ['orange','green','blue']
plot_dims = ["z", "y", "x"]
torch_model = f"{MODEL_SUB_DIR}/TP-Tf_7D_30demos_3d3e_model0_{MAX_EPOCHS}epochs.pt"
load_model_file = os.path.join(MODEL_DIR_PATH, torch_model)
saved_model = torch.load(load_model_file)['model']
saved_model.eval()
traj_index = -1
for i, file in enumerate(task_files):
    gt_start_pts, tf_start_pts = [], []
    for demo_input in plot_dataloader:
        obj_seq, traj_seq, traj_target, _, demo_idx = demo_input

        if obj_seq[0,0,i-n_tasks]!=1: continue # select only input for the i-th task
        obj_seq_input = obj_seq.to(device)
        traj_target_input = traj_target.to(device)

        predicted_traj = saved_model(obj_seq_input, traj_target_input).cpu().detach().numpy()

        mask = traj_seq[0,:,n_dims]==1

        gt_start = obj_seq[0,obj_indice[i],].numpy()
        # ==============================
        tf_start = predicted_traj[0,mask,:3][traj_index,:] # select first or last via traj_index
        # if (i==1): tf_start = predicted_traj[0,mask,:3][75,:] # needs to be used to select the pouring position
        # ==============================
        gt_start_pts.append(gt_start)
        tf_start_pts.append(tf_start)
    tf_start_pts = np.array(tf_start_pts)*train_std
    gt_start_pts = np.array(gt_start_pts)
    obj_type = unique_objs[to_obj_index(gt_start_pts[0,n_dims:n_dims+n_objs])]
    gt_start_pts = gt_start_pts[:, :3]*train_std
    # tf_start_pts -= tf_start_pts.mean(axis=0)
    axs[i].plot([-3000, 3000], [-3000, 3000], ls="--", c="red", alpha=0.5, zorder=0)
    for d in range(1, 3):
        axis_sign = 1
        if d == 0:
            axis_sign = -1
        axs[i].scatter(x = axis_sign*gt_start_pts[:,d], y = axis_sign*tf_start_pts[:,d], s=96, c=plot_colors[d],
                       label = f"{plot_dims[d]}-dimension")

    axs[i].set_xlabel(f"{obj_type} position (mm)")
    axs[i].set_ylabel(f"prediction (mm)")
    axs[i].set_xlim(lims[0], lims[1])
    axs[i].set_ylim(lims[0], lims[1])
    # axs[i].set_title(f"{task_names[i]}")
    axs[i].locator_params(axis='both', nbins=4)
    axs[i].set_yticklabels(axs[i].get_yticks(), rotation = 90)

# plt.legend()
plt.savefig(f'./graphs/tp_correlation_{"end" if traj_index else "start"}.svg', format='svg', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 14})
fig = plt.figure(figsize = (6, 6))
ax = fig.add_subplot(1, 1, 1, projection='3d')
ax.set_facecolor('white')
ax.locator_params(nbins=3, axis='z')
colors = ['red', 'blue', 'yellow', 'orange', 'green', 'purple','pink']

plot_augmentation_type = 'rotation'
random.seed(103)
for i_data, sample_data in enumerate(valid_data + test_data):

    obj_seq, traj_seq, traj_target, _, dm_idx = sample_data
    if obj_seq[0,-1]!=1: continue

    # sim2real_array_op([traj_seq, obj_seq])
    # traj_seq = traj_seq[traj_seq[:,n_dims]==1,:3].numpy() * train_std + train_mean
    # obj_seq = obj_seq[:,:3].numpy() * train_std + train_mean
    # obj1_pos = obj_seq[1,:3]
    # obj0_pos = obj_seq[0,:3]
    # line = ax.plot(traj_seq[:,2], traj_seq[:,1], -traj_seq[:,0], '--', color=colors[i_data%len(colors)])
    obj_seq = obj_seq.unsqueeze(0).to(device)
    traj_target = traj_target.unsqueeze(0).to(device)
    predicted_traj = saved_model(obj_seq, traj_target).cpu().detach().numpy()[0,traj_seq[:,n_dims]==1,:]
    obj_seq = obj_seq.cpu()
    # predicted_traj = predicted_traj
#     traj_seq = traj_seq[traj_seq[:,n_dims]==1,:3].numpy() * train_std + train_mean
#     obj1_pos = obj_seq[1,:3] * train_std + train_mean
#     obj0_pos = obj_seq[0,:3] * train_std + train_mean
    line = ax.plot(predicted_traj[:,2], predicted_traj[:,1], -predicted_traj[:,0], '--', color=colors[i_data%len(colors)], label = f'demo {i_data}')
    center_length= 30
    ax.plot(obj_seq[0,1, 2], obj_seq[0,1, 1], -obj_seq[0,1, 0], 'o',
            color=colors[i_data%len(colors)], label=f'old cup center')
    ax.plot(obj_seq[0,0, 2], obj_seq[0,0, 1], -obj_seq[0,0, 0], 'x',
            color=colors[i_data%len(colors)], label=f'old pitcher center')

ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_zlabel('z (mm)')
plt.savefig(os.path.join(GRAPH_DIR_PATH, f'traj_{plot_augmentation_type}.svg'), format='svg', dpi=300, transparent=True)
plt.show()